# Run spadupaGNN on Kaggle (ESM-2)

Enable **GPU + Internet** in Notebook Settings. Add a secret named `HUGGINGFACE_HUB_TOKEN` if private HF downloads are needed. This notebook clones the repo, installs `fair-esm`, runs a quick ESM-2 embedding test, and then runs the real multi-enzyme comparative job for NDM, VIM, and IMP.

If you hit GPU limits, keep the family order `NDM,VIM,IMP` and reduce `--max-family-seqs` or rerun with `NDM,VIM` first.

In [ ]:
# Clone repository and check Python / GPU status
!git clone https://github.com/AHFE0924/spadupaGNN || true
%cd spadupaGNN
python - <<'PY'
import torch, sys
print('python', sys.version.split()[0])
print('torch:', getattr(torch, '__version__', 'not installed'))
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    try:
        print('gpu:', torch.cuda.get_device_name(0))
    except Exception as e:
        print('gpu name error:', e)
PY

In [ ]:
# Install dependencies (fair-esm provides esm models)
!pip install -q fair-esm
!pip install -q -r requirements.txt || true
# optional helpers
!pip install -q transformers accelerate || true

In [ ]:
# NOTE: Set HUGGINGFACE_HUB_TOKEN via Notebook Settings -> Secrets.
import os
# Example (do NOT hardcode tokens in the notebook):
# os.environ['HUGGINGFACE_HUB_TOKEN'] = 'hf_xxx'
print('Set HUGGINGFACE_HUB_TOKEN via Notebook Settings -> Add Secret')

In [ ]:
# Quick ESM-2 embedding test using esm2_t33_650M_UR50D (fair-esm)
import esm, torch, numpy as np
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device).eval()
# NDM-1 sequence (representative)
ndm1 = 'MELPNIMHPVAKLSTALAAALMLSGCMPGEIRPTIGQQMETGDQRFGDLVFRQLAPNVWQHTSYLDMPGFGAVASNGLIVRDGGRVLVVDTAWTDDQTAQILNWIKQEINLPVALAVVTHAHQDKMGGMDALHAAGIATYANALSNQLAPQEGMVAAQHSLTFAANGWVEPATAPNFGPLKVFYPGPGHTSDNITVGIDGTDIAFGGCLIKDSKAKSLGNLGDADTEHYAASARAFGAAFPKASMIVMSHSAPDSRAAITHTARMADKLR'
data = [('NDM1', ndm1)]
labels, seqs, toks = batch_converter(data)
toks = toks.to(device)
with torch.no_grad():
    out = model(toks, repr_layers=[33], return_contacts=False)
emb = out['representations'][33][0, 1:len(ndm1)+1].cpu().numpy()
print('ESM-2 embedding shape:', emb.shape)
np.savez_compressed('ndm1_esm2_embedding.npz', emb=emb)

In [ ]:
# Run the real multi-enzyme ESM-2 pipeline on Kaggle.
# NDM uses curated resistance labels from the repo; VIM/IMP use observed variant positions
# from the homolog FASTA. If GPU memory is tight, rerun with `--family-order NDM,VIM`.
!python scripts/kaggle_multi_enzyme_real.py --input data/b1_filtered.fasta --output output/kaggle_real --device cuda --family-order NDM,VIM,IMP --max-family-seqs 16 --batch-size 2

## Notes & Troubleshooting
- If you see `transformer-engine` errors, prefer `fair-esm` (used above) instead of HF `transformers` for ESM-2.
- Enable GPU and Internet in Notebook Settings before running.
- Monitor disk usage: the `esm` checkpoints will be cached in `~/.cache` or `esm_cache/`.
- The summary CSV reports ROC-AUC for NDM against curated resistance positions, and VIM/IMP against observed variant positions in the homolog FASTA.
- If the GPU runs out of memory, rerun with `--family-order NDM,VIM` first and lower `--max-family-seqs`.